# 04 — Speaker ↔ Mouth Track Alignment

ใช้ข้อมูล:
- `dataset/mouth_crops/video001/mouth_metadata.csv`
- `dataset/transcripts/video001/transcript_with_frames.csv`

เป้าหมายคือประเมินว่า transcript แต่ละช่วงน่าจะเป็นของ `track_id` ไหน โดยดู **lip motion** ของ mouth crops

ผลลัพธ์:
- `speaker_track_alignment.csv`
- `speaker_track_alignment_auto.csv`
- `speaker_track_alignment_review.csv`
- `mouth_motion.csv`
- `segment_track_activity.csv`

> ช่วงที่ไม่มั่นใจจะถูกทำเครื่องหมาย `REVIEW` แทนการบังคับ label

## Cell 1 — ตั้งค่า Path

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

MOUTH_META = PROJECT_ROOT / "dataset" / "mouth_crops" / VIDEO_ID / "mouth_metadata.csv"
TRANSCRIPT_CSV = PROJECT_ROOT / "dataset" / "transcripts" / VIDEO_ID / "transcript_with_frames.csv"

OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "speaker_alignment" / VIDEO_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("MOUTH_META     :", MOUTH_META)
print("TRANSCRIPT_CSV :", TRANSCRIPT_CSV)
print("OUTPUT_ROOT    :", OUTPUT_ROOT)

if not MOUTH_META.exists():
    raise FileNotFoundError(MOUTH_META)

if not TRANSCRIPT_CSV.exists():
    raise FileNotFoundError(TRANSCRIPT_CSV)

## Cell 2 — โหลดข้อมูล

In [ ]:
import pandas as pd

mouth_df = pd.read_csv(MOUTH_META)
transcript_df = pd.read_csv(TRANSCRIPT_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = (
        mouth_df["accepted"].astype(str).str.lower().isin(["true", "1", "yes"])
    )

mouth_df = mouth_df[mouth_df["accepted"] == True].copy()
mouth_df["track_id"] = mouth_df["track_id"].astype(int)
mouth_df["frame_index"] = mouth_df["frame_index"].astype(int)

transcript_df["start_frame"] = transcript_df["start_frame"].astype(int)
transcript_df["end_frame"] = transcript_df["end_frame"].astype(int)

print("Mouth rows      :", len(mouth_df))
print("Track IDs       :", sorted(mouth_df["track_id"].unique().tolist()))
print("Transcript rows :", len(transcript_df))

display(mouth_df.head())
display(transcript_df.head())

## Cell 3 — ตรวจ Coverage

In [ ]:
max_mouth_frame = int(mouth_df["frame_index"].max()) if len(mouth_df) else -1
max_transcript_frame = int(transcript_df["end_frame"].max()) if len(transcript_df) else -1

coverage = (
    max_mouth_frame / max_transcript_frame
    if max_transcript_frame > 0 else 0
)

print("Max mouth frame     :", max_mouth_frame)
print("Max transcript frame:", max_transcript_frame)
print("Approx coverage     :", f"{coverage:.2%}")

if coverage < 0.90:
    print(
        "\n[WARNING] mouth crops ยังไม่ครอบคลุมทั้งวิดีโอ "
        "แนะนำให้รัน Face Detection + Mouth Extraction ทั้งคลิปก่อน"
    )

## Cell 4 — ฟังก์ชันอ่าน Mouth + Motion

In [ ]:
import cv2
import numpy as np

def resolve_mouth_path(relative_path):
    if not isinstance(relative_path, str) or not relative_path.strip():
        return None

    path = Path(relative_path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def read_mouth_gray(relative_path):
    path = resolve_mouth_path(relative_path)

    if path is None or not path.exists():
        return None

    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        return None

    if image.shape != (96, 96):
        image = cv2.resize(image, (96, 96), interpolation=cv2.INTER_AREA)

    return image


def frame_difference(img_a, img_b):
    if img_a is None or img_b is None:
        return np.nan

    return float(cv2.absdiff(img_a, img_b).mean() / 255.0)

print("Motion functions ready")

## Cell 5 — คำนวณ Motion ต่อ Frame

In [ ]:
from tqdm.auto import tqdm

motion_rows = []

for track_id, group in mouth_df.groupby("track_id"):
    group = group.sort_values("frame_index").reset_index(drop=True)

    prev_image = None
    prev_frame = None

    for _, row in tqdm(
        group.iterrows(),
        total=len(group),
        desc=f"track {track_id}"
    ):
        current_image = read_mouth_gray(row["mouth_file"])
        current_frame = int(row["frame_index"])

        if (
            prev_image is not None
            and prev_frame is not None
            and current_frame - prev_frame <= 2
        ):
            motion = frame_difference(prev_image, current_image)
        else:
            motion = np.nan

        motion_rows.append({
            "track_id": int(track_id),
            "frame_index": current_frame,
            "motion": motion,
            "blur_score": row.get("blur_score", np.nan),
            "brightness": row.get("brightness", np.nan),
        })

        prev_image = current_image
        prev_frame = current_frame

motion_df = pd.DataFrame(motion_rows)

print("Motion rows:", len(motion_df))
display(motion_df.head(20))

## Cell 6 — Smooth Motion

In [ ]:
motion_df = motion_df.sort_values(
    ["track_id", "frame_index"]
).reset_index(drop=True)

motion_df["motion_smooth"] = (
    motion_df
    .groupby("track_id")["motion"]
    .transform(
        lambda s: s.rolling(
            window=5,
            center=True,
            min_periods=1
        ).median()
    )
)

display(motion_df.head(20))

## Cell 7 — Activity ต่อ Transcript Segment

In [ ]:
segment_activity_rows = []

for _, seg in tqdm(
    transcript_df.iterrows(),
    total=len(transcript_df),
    desc="segments"
):
    segment_id = int(seg["segment_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])

    expected_frames = max(1, end_frame - start_frame + 1)

    matched = motion_df[
        (motion_df["frame_index"] >= start_frame)
        &
        (motion_df["frame_index"] <= end_frame)
    ]

    if len(matched) == 0:
        segment_activity_rows.append({
            "segment_id": segment_id,
            "track_id": None,
            "mean_motion": np.nan,
            "median_motion": np.nan,
            "peak_motion": np.nan,
            "coverage": 0.0,
            "observed_frames": 0,
        })
        continue

    for track_id, track_data in matched.groupby("track_id"):
        valid_motion = track_data["motion_smooth"].dropna()
        observed_frames = int(track_data["frame_index"].nunique())

        segment_activity_rows.append({
            "segment_id": segment_id,
            "track_id": int(track_id),
            "mean_motion": float(valid_motion.mean()) if len(valid_motion) else np.nan,
            "median_motion": float(valid_motion.median()) if len(valid_motion) else np.nan,
            "peak_motion": float(valid_motion.quantile(0.90)) if len(valid_motion) else np.nan,
            "coverage": min(1.0, observed_frames / expected_frames),
            "observed_frames": observed_frames,
        })

activity_df = pd.DataFrame(segment_activity_rows)
display(activity_df.head(30))

## Cell 8 — Speaker Score

In [ ]:
activity_df["mean_motion"] = activity_df["mean_motion"].fillna(0.0)
activity_df["peak_motion"] = activity_df["peak_motion"].fillna(0.0)
activity_df["coverage"] = activity_df["coverage"].fillna(0.0)

activity_df["speaker_score"] = (
    0.50 * activity_df["mean_motion"]
    + 0.30 * activity_df["peak_motion"]
    + 0.20 * activity_df["coverage"]
)

display(
    activity_df.sort_values(
        ["segment_id", "speaker_score"],
        ascending=[True, False]
    ).head(30)
)

## Cell 9 — เลือก Candidate Speaker

In [ ]:
MIN_COVERAGE = 0.50
MIN_ACTIVITY = 0.015
AUTO_MARGIN = 0.20

alignment_rows = []

for _, seg in transcript_df.iterrows():
    segment_id = int(seg["segment_id"])

    candidates = activity_df[
        activity_df["segment_id"] == segment_id
    ].copy()

    candidates = candidates[candidates["track_id"].notna()]
    candidates = candidates.sort_values("speaker_score", ascending=False)

    if len(candidates) == 0:
        alignment_rows.append({
            "video_id": VIDEO_ID,
            "segment_id": segment_id,
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "start_frame": int(seg["start_frame"]),
            "end_frame": int(seg["end_frame"]),
            "text": seg["text"],
            "candidate_track_id": None,
            "speaker_score": 0.0,
            "second_score": 0.0,
            "margin": 0.0,
            "coverage": 0.0,
            "status": "NO_DATA",
        })
        continue

    top = candidates.iloc[0]
    second_score = (
        float(candidates.iloc[1]["speaker_score"])
        if len(candidates) > 1 else 0.0
    )

    top_score = float(top["speaker_score"])
    margin = (top_score - second_score) / max(top_score, 1e-8)
    coverage_value = float(top["coverage"])

    if coverage_value < MIN_COVERAGE:
        status = "NO_DATA"
    elif top_score < MIN_ACTIVITY:
        status = "REVIEW"
    elif len(candidates) > 1 and margin < AUTO_MARGIN:
        status = "REVIEW"
    else:
        status = "AUTO"

    alignment_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "start_time": seg["start_time"],
        "end_time": seg["end_time"],
        "start_frame": int(seg["start_frame"]),
        "end_frame": int(seg["end_frame"]),
        "text": seg["text"],
        "candidate_track_id": int(top["track_id"]),
        "speaker_score": round(top_score, 6),
        "second_score": round(second_score, 6),
        "margin": round(float(margin), 6),
        "coverage": round(coverage_value, 6),
        "status": status,
    })

alignment_df = pd.DataFrame(alignment_rows)
display(alignment_df.head(30))

## Cell 10 — สรุปผล

In [ ]:
print("Total segments:", len(alignment_df))
print()
print(alignment_df["status"].value_counts(dropna=False))

print("\nAUTO tracks:")
display(
    alignment_df[
        alignment_df["status"] == "AUTO"
    ]["candidate_track_id"].value_counts()
)

## Cell 11 — Export

In [ ]:
ALL_CSV = OUTPUT_ROOT / "speaker_track_alignment.csv"
AUTO_CSV = OUTPUT_ROOT / "speaker_track_alignment_auto.csv"
REVIEW_CSV = OUTPUT_ROOT / "speaker_track_alignment_review.csv"
MOTION_CSV = OUTPUT_ROOT / "mouth_motion.csv"
ACTIVITY_CSV = OUTPUT_ROOT / "segment_track_activity.csv"

alignment_df.to_csv(ALL_CSV, index=False, encoding="utf-8-sig")

alignment_df[
    alignment_df["status"] == "AUTO"
].to_csv(
    AUTO_CSV,
    index=False,
    encoding="utf-8-sig"
)

alignment_df[
    alignment_df["status"] != "AUTO"
].to_csv(
    REVIEW_CSV,
    index=False,
    encoding="utf-8-sig"
)

motion_df.to_csv(MOTION_CSV, index=False, encoding="utf-8-sig")
activity_df.to_csv(ACTIVITY_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", ALL_CSV)
print(" -", AUTO_CSV)
print(" -", REVIEW_CSV)
print(" -", MOTION_CSV)
print(" -", ACTIVITY_CSV)

## Cell 12 — Preview REVIEW

In [ ]:
review_df = alignment_df[
    alignment_df["status"] != "AUTO"
].copy()

display(
    review_df[
        [
            "segment_id",
            "start_time",
            "end_time",
            "text",
            "candidate_track_id",
            "speaker_score",
            "margin",
            "coverage",
            "status",
        ]
    ].head(50)
)

# ขั้นต่อไป — 05 Labeled Lip Clip Builder

เมื่อ alignment ดีพอ เราจะ:
1. ใช้เฉพาะ `AUTO` ก่อน
2. ดึง mouth frames ตาม start/end frame
3. reject segment ที่เฟรมขาดเยอะ
4. สร้าง clip และ manifest
5. เตรียม train / val / test